# 3.1a Deploying the fraud model (and where foundation models run)

**Story:** The SafeDrive fraud model is trained. Claims handlers need a score while they talk to the
customer, the finance team wants every claim re-scored each night, and the GenAI team asks where
their foundation model (FM) should run. You pick the deployment target for each need.

**Task 3.1:** Manage deployment infrastructure for ML and AI model types.

| Skill | What you do here | Status |
|---|---|---|
| 3.1.1 | Compute and deployment targets: decision table | ✅ |
| 3.1.3 | Serverless, real-time, asynchronous and batch inference | ✅ |
| 3.1.2 | Multi-model endpoint, serial inference pipeline, orchestrator table | ✅ |
| 3.1.5 | Model trained outside AWS on SageMaker; Bedrock Custom Model Import 🔒 | ✅ |
| 3.1.4 | FM options: on-demand, cross-region profile, Provisioned Throughput, Marketplace, JumpStart 🔒 | ✅ |
| 3.1.7 | Inference components: copies, CPU, memory, accelerators | ✅ |

Run time is about 40 minutes (each instance endpoint takes about 5 minutes to start).
💰 Instance endpoints bill per hour, so every section deletes its endpoint at the end.

## Key points for the exam

- **Real-time endpoint:** always-on instances, low latency, payload up to 25 MB, 60 s timeout.
- **Serverless endpoint:** no instances to manage, scales to zero, cold starts, CPU only, payload up to 4 MB.
- **Asynchronous endpoint:** requests are queued, input and output are in S3, payload up to 1 GB,
  up to 1 hour per request, can scale to zero instances.
- **Batch transform:** a job, not an endpoint. Scores whole S3 files and then stops.
- **Multi-model endpoint (MME):** many models, one container, one endpoint. Models load on demand.
- **Multi-container endpoint:** up to 15 different containers. `Serial` = inference pipeline, `Direct` = pick one.
- **Inference components:** several models share the instances of one endpoint, each with its own CPU, memory, GPU and copy count.
- **Bedrock:** on-demand (pay per token), cross-region inference profile (more capacity),
  Provisioned Throughput (reserved capacity, needed for most custom models).

| Need | Choose |
|---|---|
| Steady traffic, millisecond latency | Real-time endpoint |
| Spiky or rare traffic, cold start is fine | Serverless endpoint |
| Large payload or long processing, near real-time | Asynchronous endpoint |
| Score a full dataset on a schedule | Batch transform |

## Setup

In [ ]:
import sys; sys.path.append("..")
import json, os, time
import pandas as pd
from botocore.exceptions import ClientError
from common import utils as U, catchup as C
os.makedirs("scripts", exist_ok=True)
print(U.REGION, U.bucket())

**Catch-up:** this notebook needs the trained fraud model. The cell returns at once if notebook 2.2a already trained it.

In [ ]:
MODEL_DATA = C.ensure_fraud_model()
state = U.load_state()
IMAGE, ROLE = state["xgb_image"], U.role_arn()
print(MODEL_DATA)
print(IMAGE)

Clients used in this notebook: `sagemaker` (control plane) and `sagemaker-runtime` (invoke).

In [ ]:
sm, smr, s3 = U.client("sagemaker"), U.client("sagemaker-runtime"), U.client("s3")
BUCKET = U.bucket()
print(sm.meta.endpoint_url, smr.meta.endpoint_url, sep="\n")

Sample requests: the test set has the label in column 0 and the 27 features after it.
The XGBoost container expects CSV rows with features only.

In [ ]:
TEST_KEY = state["test_s3"].split(f"{BUCKET}/", 1)[1]
test = U.read_csv(TEST_KEY, header=None)
labels, features = test.iloc[:, 0], test.iloc[:, 1:]
payload = features.head(5).to_csv(header=False, index=False).strip()
print(features.shape)
print(payload.splitlines()[0])

Small helpers used by every section: "does it exist", endpoint status, and "score these CSV rows".

In [ ]:
def exists(fn, **kwargs):
    try:
        fn(**kwargs)
        return True
    except ClientError:
        return False

def endpoint_status(name):
    try:
        return sm.describe_endpoint(EndpointName=name)["EndpointStatus"]
    except ClientError:
        return None

def score(endpoint, body, **kwargs):
    resp = smr.invoke_endpoint(EndpointName=endpoint, ContentType="text/csv", Body=body, **kwargs)
    text = resp["Body"].read().decode().replace("\n", ",")
    return [round(float(x), 4) for x in text.split(",") if x.strip()]

print("helpers ready")

## 3.1.1 Compute environments and deployment targets

- Start with the most managed option that meets the need. Move down only for a clear reason.
- CPU is enough for tree models such as XGBoost. GPUs are for deep learning and FMs.
- SageMaker endpoints give you model hosting features (variants, auto scaling, Model Monitor) for free.

| Target | Choose it when | Watch out |
|---|---|---|
| SageMaker real-time endpoint | Steady traffic, low latency, need GPU or VPC | Pay per instance hour, even when idle |
| SageMaker serverless endpoint | Spiky or low traffic | Cold start, no GPU, no VPC config, 6 GB memory max |
| SageMaker asynchronous endpoint | Payload up to 1 GB, minutes of processing | Result comes later in S3 (SNS can notify) |
| SageMaker batch transform | Offline scoring of files | No endpoint, not for live requests |
| AWS Lambda | Small model, event driven, glue code | 15 min, 10 GB image, no GPU |
| Amazon ECS / EKS (Fargate or EC2) | Team already runs containers or Kubernetes, custom serving stack | You manage scaling and deployment |
| Amazon EC2 | Full control, special drivers or licences | You manage everything |
| Amazon Bedrock | Foundation models through an API, no infrastructure | Model list and quotas are set by AWS |
| SageMaker JumpStart | Open-weight FM on your own endpoint, full control | GPU instances bill per hour |
| Edge (IoT Greengrass) | Inference on a device with no network | Model must be small or compiled |

This cell is free: it lists the endpoints that exist now, so you can see what is already deployed.

In [ ]:
endpoints = sm.list_endpoints(NameContains="safedrive")["Endpoints"]
pd.DataFrame(endpoints, columns=["EndpointName", "EndpointStatus", "CreationTime"])

## 3.1.3 Inference strategies: serverless endpoint

- Hosting always has three resources: **model** (image + artifact + role), **endpoint config** (how to host), **endpoint** (the URL).
- A serverless variant has `ServerlessConfig` (memory and max concurrency) and no instance type.
- The endpoint is named `safedrive-fraud`. Later notebooks use it, so we reuse it if it already exists.

In [ ]:
ENDPOINT = C.ENDPOINT_NAME
need_serverless = endpoint_status(ENDPOINT) is None
print(ENDPOINT, "->", endpoint_status(ENDPOINT), "| create it now:", need_serverless)

Step 1: create the **model**. It only links the container image, the model artifact in S3 and the IAM role.

In [ ]:
if need_serverless:
    sl_name = f"safedrive-fraud-{int(time.time())}"
    sm.create_model(ModelName=sl_name, ExecutionRoleArn=ROLE, Tags=U.TAGS,
                    PrimaryContainer={"Image": IMAGE, "ModelDataUrl": MODEL_DATA})
    print("created model", sl_name)
else:
    print("endpoint exists - nothing to create")

Step 2: create the **endpoint config**. `ServerlessConfig` is what makes it serverless.

In [ ]:
if need_serverless:
    sm.create_endpoint_config(
        EndpointConfigName=sl_name, Tags=U.TAGS,
        ProductionVariants=[{"VariantName": "AllTraffic", "ModelName": sl_name,
                             "ServerlessConfig": {"MemorySizeInMB": 2048, "MaxConcurrency": 5}}])
    print("created endpoint config", sl_name)
else:
    print("endpoint exists - nothing to create")

Step 3: create the **endpoint** and save its name in the shared state.

In [ ]:
if need_serverless:
    sm.create_endpoint(EndpointName=ENDPOINT, EndpointConfigName=sl_name, Tags=U.TAGS)
    U.save_state(fraud_endpoint=ENDPOINT, fraud_model_name=sl_name)
print(ENDPOINT, "->", endpoint_status(ENDPOINT))

Wait until the endpoint is `InService` (2 to 4 minutes for a new endpoint).

In [ ]:
sm.get_waiter("endpoint_in_service").wait(EndpointName=ENDPOINT)
need_serverless = False
print(ENDPOINT, "->", endpoint_status(ENDPOINT))

Look at the config behind the endpoint: there is no instance type, only memory and concurrency.

In [ ]:
config_name = sm.describe_endpoint(EndpointName=ENDPOINT)["EndpointConfigName"]
sm.describe_endpoint_config(EndpointConfigName=config_name)["ProductionVariants"]

Invoke it with 5 claims. The answer is one fraud probability per row.

In [ ]:
scores = score(ENDPOINT, payload)
pd.DataFrame({"actual_fraud": labels.head(5).values, "fraud_probability": scores})

Cold start check: time a few calls. The first call after idle time is slower, the next ones are fast.

In [ ]:
for i in range(4):
    start = time.time()
    score(ENDPOINT, payload)
    print(f"call {i + 1}: {(time.time() - start) * 1000:.0f} ms")

## 3.1.3 Inference strategies: real-time endpoint

- Same three resources. The variant now has an **instance type and count**.
- Instances are always on: lowest latency, but you pay while it is idle.
- One SageMaker model (`safedrive-fraud-xgb`) is reused by the real-time, async, batch and inference-component sections.

In [ ]:
MODEL = "safedrive-fraud-xgb"
if not exists(sm.describe_model, ModelName=MODEL):
    sm.create_model(ModelName=MODEL, ExecutionRoleArn=ROLE, Tags=U.TAGS,
                    PrimaryContainer={"Image": IMAGE, "ModelDataUrl": MODEL_DATA})
sm.describe_model(ModelName=MODEL)["PrimaryContainer"]

Create the endpoint config: one `ml.m5.large` instance.

In [ ]:
RT = "safedrive-fraud-rt"
if not exists(sm.describe_endpoint_config, EndpointConfigName=RT):
    sm.create_endpoint_config(
        EndpointConfigName=RT, Tags=U.TAGS,
        ProductionVariants=[{"VariantName": "AllTraffic", "ModelName": MODEL,
                             "InstanceType": "ml.m5.large", "InitialInstanceCount": 1}])
print("endpoint config:", RT)

Create the endpoint (💰 about $0.12 per hour until deleted).

In [ ]:
if endpoint_status(RT) is None:
    sm.create_endpoint(EndpointName=RT, EndpointConfigName=RT, Tags=U.TAGS)
print(RT, "->", endpoint_status(RT))

Wait for `InService` (about 4 to 6 minutes: SageMaker starts the instance and pulls the image).

In [ ]:
sm.get_waiter("endpoint_in_service").wait(EndpointName=RT)
print(RT, "->", endpoint_status(RT))

Invoke and time it. Compare the latency with the serverless calls above (no cold start here).

In [ ]:
for i in range(3):
    start = time.time()
    scores = score(RT, payload)
    print(f"call {i + 1}: {(time.time() - start) * 1000:.0f} ms", scores)

Delete the real-time endpoint now. The endpoint is what costs money; the config and model are free.

In [ ]:
U.safe(sm.delete_endpoint, EndpointName=RT)
U.safe(sm.delete_endpoint_config, EndpointConfigName=RT)

## 3.1.3 Inference strategies: asynchronous endpoint

- The request is a **pointer to an S3 object**. SageMaker queues it and writes the result to S3.
- Use it for big payloads (up to 1 GB) or slow models (up to 1 hour).
- With auto scaling it can go down to **0 instances** when the queue is empty.
- Optional SNS topics tell you about success or failure.

The endpoint config has an extra block: `AsyncInferenceConfig`.

In [ ]:
ASYNC = "safedrive-fraud-async"
if not exists(sm.describe_endpoint_config, EndpointConfigName=ASYNC):
    sm.create_endpoint_config(
        EndpointConfigName=ASYNC, Tags=U.TAGS,
        ProductionVariants=[{"VariantName": "AllTraffic", "ModelName": MODEL,
                             "InstanceType": "ml.m5.large", "InitialInstanceCount": 1}],
        AsyncInferenceConfig={
            "OutputConfig": {"S3OutputPath": U.s3_uri("safedrive/async/output"),
                             "S3FailurePath": U.s3_uri("safedrive/async/failure")},
            "ClientConfig": {"MaxConcurrentInvocationsPerInstance": 4}})
print("endpoint config:", ASYNC)

Create the asynchronous endpoint.

In [ ]:
if endpoint_status(ASYNC) is None:
    sm.create_endpoint(EndpointName=ASYNC, EndpointConfigName=ASYNC, Tags=U.TAGS)
print(ASYNC, "->", endpoint_status(ASYNC))

While it starts, upload the request file: 200 claims as CSV.

In [ ]:
ASYNC_INPUT = "safedrive/async/input/claims_batch.csv"
body = features.head(200).to_csv(header=False, index=False)
s3.put_object(Bucket=BUCKET, Key=ASYNC_INPUT, Body=body)
print(U.s3_uri(ASYNC_INPUT), len(body), "bytes")

Wait for `InService` (about 4 to 6 minutes).

In [ ]:
sm.get_waiter("endpoint_in_service").wait(EndpointName=ASYNC)
print(ASYNC, "->", endpoint_status(ASYNC))

Invoke with `invoke_endpoint_async`. You get back at once with the S3 location where the answer **will** be.

In [ ]:
resp = smr.invoke_endpoint_async(EndpointName=ASYNC, InputLocation=U.s3_uri(ASYNC_INPUT),
                                 ContentType="text/csv")
print("inference id:", resp["InferenceId"])
print("output      :", resp["OutputLocation"])

Poll S3 until the output object exists.

In [ ]:
out_key = resp["OutputLocation"].split(f"{BUCKET}/", 1)[1]
U.wait_until(lambda: exists(s3.head_object, Bucket=BUCKET, Key=out_key),
             ok=(True,), delay=3, timeout=300, label="output ready")

Read the result: 200 probabilities.

In [ ]:
text = U.read_bytes(out_key).decode().replace("\n", ",")
async_scores = [float(x) for x in text.split(",") if x.strip()]
print(len(async_scores), "scores; first 5:", [round(x, 4) for x in async_scores[:5]])

Delete the asynchronous endpoint.

In [ ]:
U.safe(sm.delete_endpoint, EndpointName=ASYNC)
U.safe(sm.delete_endpoint_config, EndpointConfigName=ASYNC)

## 3.1.3 Inference strategies: batch transform

- Nightly claims scoring: no endpoint. A job starts instances, scores the files, writes results to S3 and stops.
- `SplitType=Line` sends the file line by line (in mini-batches with `BatchStrategy=MultiRecord`).
- `DataProcessing` filters and joins columns:
  `InputFilter` drops the label before scoring, `JoinSource=Input` puts the score next to the input row,
  `OutputFilter` keeps only the columns you want.

Start the job on the test set. Output will have 2 columns: the label and the score.

In [ ]:
JOB = f"safedrive-fraud-batch-{int(time.time())}"
sm.create_transform_job(
    TransformJobName=JOB, ModelName=MODEL, BatchStrategy="MultiRecord", MaxPayloadInMB=6, Tags=U.TAGS,
    TransformInput={"DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": state["test_s3"]}},
                    "ContentType": "text/csv", "SplitType": "Line"},
    TransformOutput={"S3OutputPath": U.s3_uri("safedrive/batch/output"),
                     "AssembleWith": "Line", "Accept": "text/csv"},
    TransformResources={"InstanceType": "ml.m5.large", "InstanceCount": 1},
    DataProcessing={"InputFilter": "$[1:]", "JoinSource": "Input", "OutputFilter": "$[0,-1]"})
print("started", JOB)

Wait for the job (about 5 minutes; most of it is instance start-up).

In [ ]:
U.wait_until(lambda: sm.describe_transform_job(TransformJobName=JOB)["TransformJobStatus"],
             ok=("Completed",), bad=("Failed", "Stopped"), delay=30, label="transform job")

Read the output. Each input file `x.csv` gives an output file `x.csv.out`.

In [ ]:
out_keys = U.list_keys("safedrive/batch/output/")
batch = U.read_csv(out_keys[0], header=None, names=["is_fraud", "fraud_probability"])
print(out_keys)
batch.head()

Because the label is next to the score, checking model quality on the batch is one line.

In [ ]:
from sklearn.metrics import roc_auc_score
print("rows:", len(batch), "| AUC:", round(roc_auc_score(batch["is_fraud"], batch["fraud_probability"]), 4))

## 3.1.2 Multi-model endpoint

- One container, one endpoint, **many model artifacts** under one S3 prefix.
- The caller picks the model with `TargetModel`. Models are loaded on first use and unloaded when memory is full.
- Good for many similar models with uneven traffic (for example one fraud model per country).
- All models must use the **same container**. Not supported on serverless endpoints.

Copy two model artifacts under one prefix. We use the tuned model from notebook 2.2a if it exists.

In [ ]:
from urllib.parse import urlparse
MME_PREFIX = "safedrive/mme"
MODEL_V2 = state.get("fraud_model_data_v2") or MODEL_DATA

def copy_artifact(uri, name):
    src = urlparse(uri)
    s3.copy_object(Bucket=BUCKET, Key=f"{MME_PREFIX}/{name}",
                   CopySource={"Bucket": src.netloc, "Key": src.path.lstrip("/")})

copy_artifact(MODEL_DATA, "fraud-v1.tar.gz")
copy_artifact(MODEL_V2, "fraud-v2.tar.gz")
U.list_keys(MME_PREFIX)

Create the model. Two things are different: `Mode="MultiModel"` and `ModelDataUrl` is a **prefix**, not a file.

In [ ]:
MME = "safedrive-fraud-mme"
if not exists(sm.describe_model, ModelName=MME):
    sm.create_model(ModelName=MME, ExecutionRoleArn=ROLE, Tags=U.TAGS,
                    PrimaryContainer={"Image": IMAGE, "Mode": "MultiModel",
                                      "ModelDataUrl": U.s3_uri(MME_PREFIX + "/")})
sm.describe_model(ModelName=MME)["PrimaryContainer"]

Create the endpoint config and the endpoint (instance based).

In [ ]:
if not exists(sm.describe_endpoint_config, EndpointConfigName=MME):
    sm.create_endpoint_config(
        EndpointConfigName=MME, Tags=U.TAGS,
        ProductionVariants=[{"VariantName": "AllTraffic", "ModelName": MME,
                             "InstanceType": "ml.m5.large", "InitialInstanceCount": 1}])
if endpoint_status(MME) is None:
    sm.create_endpoint(EndpointName=MME, EndpointConfigName=MME, Tags=U.TAGS)
print(MME, "->", endpoint_status(MME))

Wait for `InService` (about 4 to 6 minutes).

In [ ]:
sm.get_waiter("endpoint_in_service").wait(EndpointName=MME)
print(MME, "->", endpoint_status(MME))

Invoke each model with `TargetModel`. The first call to a model is slower because it is loaded from S3.

In [ ]:
for target in ["fraud-v1.tar.gz", "fraud-v2.tar.gz", "fraud-v1.tar.gz"]:
    start = time.time()
    scores = score(MME, payload, TargetModel=target)
    print(f"{target}: {(time.time() - start) * 1000:.0f} ms", scores[:3])

Add a third model **without redeploying**: copy a new artifact under the prefix and call it.

In [ ]:
copy_artifact(MODEL_DATA, "fraud-uk.tar.gz")
print("fraud-uk.tar.gz:", score(MME, payload, TargetModel="fraud-uk.tar.gz")[:3])

Delete the multi-model endpoint.

In [ ]:
U.safe(sm.delete_endpoint, EndpointName=MME)
U.safe(sm.delete_endpoint_config, EndpointConfigName=MME)
U.safe(sm.delete_model, ModelName=MME)

## 3.1.2 Multi-container endpoints and the serial inference pipeline

| Pattern | What it is | Choose it when |
|---|---|---|
| Multi-model endpoint | Many artifacts, one container | Many models, same framework |
| Multi-container, `Direct` | Up to 15 containers, caller picks one with `TargetContainerHostname` | Few models, different frameworks |
| Multi-container, `Serial` (inference pipeline) | 2 to 15 containers in a chain; output of one is input of the next | Preprocess, predict, postprocess in one call |
| Inference components | Models share instances with their own resources and scaling | FMs and mixed models on shared GPUs |
| Production variants | Several models behind one endpoint with traffic weights | A/B tests, shadow tests |

Hands-on: a 2-step pipeline. Container 1 turns a JSON claim (named features) into the CSV row that
container 2 (XGBoost) expects. First write the script for container 1.

In [ ]:
%%writefile scripts/pipeline_preprocess.py
"""Container 1 of the inference pipeline: JSON with named features -> CSV row in model order."""
import json
import os


def model_fn(model_dir):
    with open(os.path.join(model_dir, "feature_names.json")) as f:
        return json.load(f)


def input_fn(body, content_type):
    data = json.loads(body)
    return data if isinstance(data, list) else [data]


def predict_fn(records, feature_names):
    return [[float(rec.get(name, 0)) for name in feature_names] for rec in records]


def output_fn(rows, accept):
    csv = "\n".join(",".join(str(v) for v in row) for row in rows)
    return csv, "text/csv"      # the content type is passed to the next container

The "model" of container 1 is just the ordered list of feature names. Package it as `model.tar.gz` and upload it.

In [ ]:
import tarfile
os.makedirs("scripts/pre_model", exist_ok=True)
with open("scripts/pre_model/feature_names.json", "w") as f:
    json.dump(state["feature_names"], f)
with tarfile.open("scripts/pre_model.tar.gz", "w:gz") as tar:
    tar.add("scripts/pre_model/feature_names.json", arcname="feature_names.json")
PRE_KEY = "safedrive/pipeline-model/pre_model.tar.gz"
s3.upload_file("scripts/pre_model.tar.gz", BUCKET, PRE_KEY)
print(U.s3_uri(PRE_KEY))

Define the two containers with the SageMaker SDK and chain them in a `PipelineModel`.

In [ ]:
import sagemaker
from sagemaker.model import Model
from sagemaker.pipeline import PipelineModel
from sagemaker.sklearn.model import SKLearnModel
sess = sagemaker.Session(boto_session=U.session, default_bucket=BUCKET)

pre_model = SKLearnModel(model_data=U.s3_uri(PRE_KEY), role=ROLE, sagemaker_session=sess,
                         entry_point="scripts/pipeline_preprocess.py", framework_version="1.2-1",
                         env={"SAGEMAKER_DEFAULT_INVOCATIONS_ACCEPT": "text/csv"})
xgb_model = Model(image_uri=IMAGE, model_data=MODEL_DATA, role=ROLE, sagemaker_session=sess)
SERIAL = "safedrive-fraud-serial"
pipeline_model = PipelineModel(models=[pre_model, xgb_model], role=ROLE, sagemaker_session=sess, name=SERIAL)
print("containers:", len(pipeline_model.models))

Deploy the pipeline to one endpoint (about 5 to 7 minutes). Both containers run on the same instance.

In [ ]:
if endpoint_status(SERIAL) is None:
    pipeline_model.deploy(initial_instance_count=1, instance_type="ml.m5.large",
                          endpoint_name=SERIAL, tags=U.TAGS)
print(SERIAL, "->", endpoint_status(SERIAL))

The SageMaker model behind it has a list of `Containers` (not one `PrimaryContainer`) and mode `Serial`.

In [ ]:
desc = sm.describe_model(ModelName=SERIAL)
print("mode:", desc.get("InferenceExecutionConfig", {"Mode": "Serial (default)"}))
[c["Image"].split("/")[-1] for c in desc["Containers"]]

Invoke with a JSON claim. The caller no longer needs to know the column order.

In [ ]:
claim = dict(zip(state["feature_names"], features.iloc[0].tolist()))
resp = smr.invoke_endpoint(EndpointName=SERIAL, ContentType="application/json", Body=json.dumps(claim))
print("sent keys:", list(claim)[:4], "...")
print("fraud probability:", resp["Body"].read().decode().strip())

Delete the pipeline endpoint and its model.

In [ ]:
U.safe(sm.delete_endpoint, EndpointName=SERIAL)
U.safe(sm.delete_endpoint_config, EndpointConfigName=SERIAL)
U.safe(sm.delete_model, ModelName=SERIAL)

For **direct** invocation the model is created like this (notes only):

```python
sm.create_model(ModelName="multi", ExecutionRoleArn=ROLE,
    Containers=[{"ContainerHostname": "xgb", "Image": xgb_image, "ModelDataUrl": xgb_artifact},
                {"ContainerHostname": "sklearn", "Image": sk_image, "ModelDataUrl": sk_artifact}],
    InferenceExecutionConfig={"Mode": "Direct"})
smr.invoke_endpoint(EndpointName="multi", TargetContainerHostname="xgb", ...)
```

## 3.1.2 Deployment orchestrators

| Orchestrator | Choose it when | Notes |
|---|---|---|
| SageMaker Pipelines | The workflow is ML steps: process, train, evaluate, register, deploy | Built into SageMaker, lineage and Model Registry, no extra cost (notebook 3.3a) |
| AWS Step Functions | The workflow mixes many AWS services, needs branching, retries, human approval | Visual state machine, direct SageMaker, Bedrock, Glue and Lambda integrations |
| Amazon MWAA (Airflow) | The team already uses Airflow DAGs, or needs non-AWS systems | Managed Airflow environment that bills per hour |
| Amazon EventBridge | You need a trigger, not a workflow: schedule or event starts something | Starts pipelines, state machines, Lambda |
| AWS CodePipeline | CI/CD: code change, build, test, deploy | Often starts a SageMaker pipeline (notebook 3.3a) |

- Exam wording: "ML-native, least effort" means SageMaker Pipelines. "Many services, complex logic" means Step Functions.
  "Existing Airflow" means MWAA. "On a schedule or on an event" means EventBridge.

## 3.1.5 Deploy a model that was built outside AWS

- SageMaker does not care where a model was trained. It needs a **container image**, a **`model.tar.gz` in S3** and an **inference script**.
- Framework containers (scikit-learn, PyTorch, TensorFlow, Hugging Face) load your script: `model_fn` loads the model, `predict_fn` scores.
- Match the framework version of the container with the one used for training, or export to a
  portable format (ONNX, JSON coefficients). A pickle from a newer scikit-learn often fails in an older container.
- If no framework container fits, bring your own container (notebook 3.2a).

Train a scikit-learn model "on the laptop" (this kernel, no SageMaker involved).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
TRAIN_KEY = state["train_s3"].split(f"{BUCKET}/", 1)[1]
train = U.read_csv(TRAIN_KEY, header=None)
scaler = StandardScaler().fit(train.iloc[:, 1:])
local_model = LogisticRegression(max_iter=500, class_weight="balanced")
local_model.fit(scaler.transform(train.iloc[:, 1:]), train.iloc[:, 0])
local_scores = local_model.predict_proba(scaler.transform(features))[:, 1]
print("local AUC:", round(roc_auc_score(labels, local_scores), 4))

Export the model in a portable way: the scaler and the coefficients as JSON. This avoids version problems with pickle files.

In [ ]:
os.makedirs("scripts/local_model", exist_ok=True)
portable = {"mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist(),
            "coef": local_model.coef_[0].tolist(), "intercept": float(local_model.intercept_[0])}
with open("scripts/local_model/model.json", "w") as f:
    json.dump(portable, f)
print({k: (len(v) if isinstance(v, list) else v) for k, v in portable.items()})

Write the inference script. The container calls `model_fn` once at start-up and `predict_fn` for every request.
CSV parsing and JSON output are handled by the container defaults.

In [ ]:
%%writefile scripts/sklearn_inference.py
"""Inference script for the model trained outside AWS (runs in the SageMaker scikit-learn container)."""
import json
import os

import numpy as np


def model_fn(model_dir):
    with open(os.path.join(model_dir, "model.json")) as f:
        m = json.load(f)
    return {k: np.array(v) for k, v in m.items()}


def predict_fn(data, model):
    x = (np.asarray(data, dtype=float) - model["mean"]) / model["scale"]
    z = x @ model["coef"] + model["intercept"]
    return 1.0 / (1.0 + np.exp(-z))

Package the artifact as `model.tar.gz` (files at the top level of the archive) and upload it.

In [ ]:
with tarfile.open("scripts/local_model.tar.gz", "w:gz") as tar:
    tar.add("scripts/local_model/model.json", arcname="model.json")
LOCAL_KEY = "safedrive/external-model/model.tar.gz"
s3.upload_file("scripts/local_model.tar.gz", BUCKET, LOCAL_KEY)
print(U.s3_uri(LOCAL_KEY))

Deploy with the SageMaker SDK: scikit-learn container + artifact + script, on a serverless endpoint (about 3 to 5 minutes).

In [ ]:
from sagemaker.serverless import ServerlessInferenceConfig
SK = "safedrive-fraud-sklearn"
sk_model = SKLearnModel(model_data=U.s3_uri(LOCAL_KEY), role=ROLE, sagemaker_session=sess,
                        entry_point="scripts/sklearn_inference.py", framework_version="1.2-1", name=SK)
if endpoint_status(SK) is None:
    sk_model.deploy(endpoint_name=SK, tags=U.TAGS,
                    serverless_inference_config=ServerlessInferenceConfig(memory_size_in_mb=2048, max_concurrency=2))
print(SK, "->", endpoint_status(SK))

Invoke it and compare with the local predictions. They should be the same numbers.

In [ ]:
resp = smr.invoke_endpoint(EndpointName=SK, ContentType="text/csv", Accept="application/json", Body=payload)
remote = json.loads(resp["Body"].read())
pd.DataFrame({"local": local_scores[:5].round(4), "endpoint": [round(x, 4) for x in remote]})

Delete the endpoint of the external model.

In [ ]:
U.safe(sm.delete_endpoint, EndpointName=SK)
U.safe(sm.delete_endpoint_config, EndpointConfigName=SK)
U.safe(sm.delete_model, ModelName=SK)

### 3.1.5 Bedrock Custom Model Import 🔒

- Brings **your own open-weight FM** (for example a Llama, Mistral, Flan or Qwen model fine-tuned elsewhere) into Bedrock.
- Input: model files in Hugging Face format (safetensors, config, tokenizer) in S3.
- Result: an imported model you call with `InvokeModel` or `Converse`. Serverless, billed per model copy while active.
- 🔒 Prerequisite: put the model weights at `IMPORT_S3_URI` first. 💰 Billed in 5-minute windows while the model is in use.

In [ ]:
bedrock, brt = U.client("bedrock"), U.client("bedrock-runtime")
IMPORT_S3_URI = U.s3_uri("safedrive/import/my-llm/")
IMPORT_JOB = "safedrive-import-job"
if U.expensive("Bedrock Custom Model Import job"):
    job = bedrock.create_model_import_job(
        jobName=IMPORT_JOB, importedModelName="safedrive-imported-llm", roleArn=ROLE,
        modelDataSource={"s3DataSource": {"s3Uri": IMPORT_S3_URI}})
    print(job["jobArn"])

🔒 Check the job, call the imported model by its ARN, then delete it.

In [ ]:
if U.expensive("call and delete the imported model"):
    job = bedrock.get_model_import_job(jobIdentifier=IMPORT_JOB)
    print(job["status"])
    if job["status"] == "Completed":
        body = json.dumps({"prompt": "What does comprehensive motor cover include?", "max_tokens": 100})
        out = brt.invoke_model(modelId=job["importedModelArn"], body=body)
        print(out["body"].read().decode()[:300])
        U.safe(bedrock.delete_imported_model, modelIdentifier=job["importedModelArn"])

## 3.1.4 Foundation model deployment options

| Option | How you pay | Choose it when |
|---|---|---|
| Bedrock **on-demand** | Per input and output token | Default. Variable or unknown traffic |
| Bedrock **cross-region inference profile** | Per token (same price) | Need more throughput; requests are routed to other Regions in the same geography (or global) |
| Bedrock **batch inference** | Per token, about 50% cheaper | Large offline jobs, no need for an immediate answer |
| Bedrock **Provisioned Throughput** 🔒 | Per model unit per hour, 1 or 6 month commitment (or none) | Steady high volume, guaranteed capacity, custom (fine-tuned) models |
| Bedrock **Marketplace** | Per instance hour (plus software fee for some models) | Model is not in the serverless list; deployed on a SageMaker-managed endpoint and called through Bedrock APIs |
| **SageMaker JumpStart** 🔒 | Per instance hour (GPU) | Open-weight model, full control of instance, VPC, container and scaling |
| **Custom Model Import** 🔒 | Per active model copy | You already have fine-tuned weights |

List the system-defined inference profiles. The ID starts with a geography (`us.`, `eu.`, `apac.`, `global.`).

In [ ]:
profiles = bedrock.list_inference_profiles(typeEquals="SYSTEM_DEFINED", maxResults=100)["inferenceProfileSummaries"]
df = pd.DataFrame(profiles)[["inferenceProfileId", "inferenceProfileName", "status"]]
print(len(df), "profiles")
df[df["inferenceProfileId"].str.contains("nova")]

Look inside the profile this course uses: it lists the Regions a request can be routed to.

In [ ]:
profile = bedrock.get_inference_profile(inferenceProfileIdentifier=U.TEXT_MODEL)
print(profile["inferenceProfileId"], "|", profile["type"])
[m["modelArn"].split(":")[3] for m in profile["models"]]

Call the model through the profile. Look at the token usage (what you pay for) and the latency.

In [ ]:
question = "In one sentence: what is a deductible in motor insurance?"
messages = [{"role": "user", "content": [{"text": question}]}]
resp = brt.converse(modelId=U.TEXT_MODEL, messages=messages, inferenceConfig={"maxTokens": 100})
print(resp["output"]["message"]["content"][0]["text"])
print(resp["usage"], "| latency ms:", resp["metrics"]["latencyMs"])

Now try plain on-demand with the base model ID (no geography prefix). In many Regions newer models
answer only through an inference profile, so an error here is expected and is itself an exam point.

In [ ]:
base_id = U.TEXT_MODEL.split(".", 1)[1]
try:
    resp = brt.converse(modelId=base_id, messages=messages, inferenceConfig={"maxTokens": 100})
    print(base_id, "->", resp["output"]["message"]["content"][0]["text"])
except ClientError as e:
    print(base_id, "->", e.response["Error"]["Message"][:200])

An **application inference profile** is your own named copy of a model or profile. It adds tags so you can
track cost and usage per application. It is free to create.

In [ ]:
APP_PROFILE = "safedrive-claims-assistant"
found = [p for p in bedrock.list_inference_profiles(typeEquals="APPLICATION")["inferenceProfileSummaries"]
         if p["inferenceProfileName"] == APP_PROFILE]
if found:
    app_arn = found[0]["inferenceProfileArn"]
else:
    app_arn = bedrock.create_inference_profile(
        inferenceProfileName=APP_PROFILE, modelSource={"copyFrom": U.model_arn(U.TEXT_MODEL)},
        tags=[{"key": "project", "value": "mla-c02-practice"}])["inferenceProfileArn"]
print(app_arn)

Call it by ARN. Same model, but the usage is now tagged to the application.

In [ ]:
resp = brt.converse(modelId=app_arn, messages=messages, inferenceConfig={"maxTokens": 100})
print(resp["output"]["message"]["content"][0]["text"])

Provisioned Throughput: this free call lists what is provisioned now (normally nothing).
The code to buy it is in notebook 3.2a (skill 3.2.1) 🔒.

In [ ]:
bedrock.list_provisioned_model_throughputs()["provisionedModelSummaries"]

### 3.1.4 SageMaker JumpStart FM on a GPU endpoint 🔒

- JumpStart has ready-made containers and settings for open-weight models. `JumpStartModel(...).deploy()` creates the endpoint.
- You choose the instance type; the model stays in your account and VPC.
- 🔒 💰 A GPU instance (`ml.g5.2xlarge`) costs about $1.50 per hour, and you need service quota for it.
  Some models also need `accept_eula=True`.

In [ ]:
JS_ENDPOINT = "safedrive-jumpstart-llm"
if U.expensive("JumpStart LLM on a GPU endpoint"):
    from sagemaker.jumpstart.model import JumpStartModel
    js_model = JumpStartModel(model_id="huggingface-llm-mistral-7b-instruct", role=ROLE,
                              sagemaker_session=sess, instance_type="ml.g5.2xlarge")
    js_predictor = js_model.deploy(endpoint_name=JS_ENDPOINT, accept_eula=True, tags=U.TAGS)
    print(js_predictor.endpoint_name)

🔒 Invoke the JumpStart endpoint, then delete it straight away.

In [ ]:
if U.expensive("invoke and delete the JumpStart endpoint"):
    answer = js_predictor.predict({"inputs": "What is a no-claims bonus?", "parameters": {"max_new_tokens": 80}})
    print(answer)
    js_predictor.delete_predictor()

## 3.1.7 FM hosting configuration and resource allocation: inference components

- An **inference component (IC)** is one model placed on an endpoint with its own resources:
  CPU cores, memory, **number of accelerators (GPUs)** and **copy count**.
- Several ICs share the same instances, so two FMs can share one multi-GPU instance instead of two endpoints.
- Each IC scales its copies on its own (can go to 0). The endpoint adds instances when copies do not fit.
- The endpoint config has **no model**. It has an execution role, instance type and a routing strategy.

| Setting | Where | Meaning |
|---|---|---|
| `NumberOfAcceleratorDevicesRequired` | IC | GPUs for one copy |
| `NumberOfCpuCoresRequired`, `MinMemoryRequiredInMb` | IC | CPU and memory for one copy |
| `CopyCount` | IC runtime config | How many copies run now |
| `ManagedInstanceScaling` | Endpoint variant | Min and max instances SageMaker may use |
| `RoutingStrategy=LEAST_OUTSTANDING_REQUESTS` | Endpoint variant | Send requests to the least busy copy (good for FMs) |

The API is the same for an FM on GPUs and for our small CPU model, so we practise with the fraud model
on one `ml.m5.large` (cheap). For an FM you would add `NumberOfAcceleratorDevicesRequired` and a GPU instance type.

Create the endpoint config: no `ModelName` in the variant.

In [ ]:
IC_ENDPOINT = "safedrive-fraud-ic"
if not exists(sm.describe_endpoint_config, EndpointConfigName=IC_ENDPOINT):
    sm.create_endpoint_config(
        EndpointConfigName=IC_ENDPOINT, ExecutionRoleArn=ROLE, Tags=U.TAGS,
        ProductionVariants=[{"VariantName": "AllTraffic", "InstanceType": "ml.m5.large",
                             "InitialInstanceCount": 1,
                             "RoutingConfig": {"RoutingStrategy": "LEAST_OUTSTANDING_REQUESTS"}}])
print("endpoint config:", IC_ENDPOINT)

Create the (empty) endpoint and wait for it (about 3 to 5 minutes).

In [ ]:
if endpoint_status(IC_ENDPOINT) is None:
    sm.create_endpoint(EndpointName=IC_ENDPOINT, EndpointConfigName=IC_ENDPOINT, Tags=U.TAGS)
sm.get_waiter("endpoint_in_service").wait(EndpointName=IC_ENDPOINT)
print(IC_ENDPOINT, "->", endpoint_status(IC_ENDPOINT))

Add inference component 1: the baseline model, 1 copy, a quarter of a CPU core and 512 MB of memory per copy.

In [ ]:
IC1, IC2 = "safedrive-fraud-ic-v1", "safedrive-fraud-ic-v2"
if not exists(sm.describe_inference_component, InferenceComponentName=IC1):
    sm.create_inference_component(
        InferenceComponentName=IC1, EndpointName=IC_ENDPOINT, VariantName="AllTraffic", Tags=U.TAGS,
        Specification={"ModelName": MODEL,
                       "ComputeResourceRequirements": {"NumberOfCpuCoresRequired": 0.25,
                                                       "MinMemoryRequiredInMb": 512}},
        RuntimeConfig={"CopyCount": 1})
print("creating", IC1)

Add inference component 2 on the **same instance**: the second model, given as a container (image + artifact) instead of a model name.

In [ ]:
if not exists(sm.describe_inference_component, InferenceComponentName=IC2):
    sm.create_inference_component(
        InferenceComponentName=IC2, EndpointName=IC_ENDPOINT, VariantName="AllTraffic", Tags=U.TAGS,
        Specification={"Container": {"Image": IMAGE, "ArtifactUrl": MODEL_V2},
                       "ComputeResourceRequirements": {"NumberOfCpuCoresRequired": 0.25,
                                                       "MinMemoryRequiredInMb": 512}},
        RuntimeConfig={"CopyCount": 1})
print("creating", IC2)

Wait until both components are `InService` (2 to 4 minutes).

In [ ]:
def ic_status(name):
    return sm.describe_inference_component(InferenceComponentName=name)["InferenceComponentStatus"]

for name in [IC1, IC2]:
    U.wait_until(lambda: ic_status(name), ok=("InService",), bad=("Failed",), delay=15, label=name)

Invoke each model with `InferenceComponentName`.

In [ ]:
for name in [IC1, IC2]:
    print(name, score(IC_ENDPOINT, payload, InferenceComponentName=name)[:3])

Resource allocation in action: ask for 2 copies of component 1. Each copy reserves its own CPU and memory on the instance.

In [ ]:
sm.update_inference_component_runtime_config(InferenceComponentName=IC1, DesiredRuntimeConfig={"CopyCount": 2})
U.wait_until(lambda: ic_status(IC1), ok=("InService",), bad=("Failed",), delay=15, label=IC1)
sm.describe_inference_component(InferenceComponentName=IC1)["RuntimeConfig"]

List what runs on the endpoint.

In [ ]:
ics = sm.list_inference_components(EndpointNameEquals=IC_ENDPOINT)["InferenceComponents"]
pd.DataFrame(ics)[["InferenceComponentName", "InferenceComponentStatus", "VariantName"]]

Delete the components first, then the endpoint.

In [ ]:
U.safe(sm.delete_inference_component, InferenceComponentName=IC1)
U.safe(sm.delete_inference_component, InferenceComponentName=IC2)

In [ ]:
U.wait_until(lambda: len(sm.list_inference_components(EndpointNameEquals=IC_ENDPOINT)["InferenceComponents"]),
             ok=(0,), delay=15, timeout=600, label="components left")
U.safe(sm.delete_endpoint, EndpointName=IC_ENDPOINT)
U.safe(sm.delete_endpoint_config, EndpointConfigName=IC_ENDPOINT)

For a real FM on GPUs the component looks like this (notes only; runnable GPU version is in notebook 3.2a, skill 3.2.10 🔒):

```python
sm.create_inference_component(
    InferenceComponentName="llm-a", EndpointName="shared-gpu", VariantName="AllTraffic",
    Specification={"ModelName": "llm-a-model",
                   "ComputeResourceRequirements": {"NumberOfAcceleratorDevicesRequired": 1,
                                                   "NumberOfCpuCoresRequired": 4,
                                                   "MinMemoryRequiredInMb": 16384}},
    RuntimeConfig={"CopyCount": 1})
```

## Exam traps

- Serverless endpoints: no GPU, no multi-model, no VPC config, 4 MB payload. If the question says GPU, it is not serverless.
- "Payload of 500 MB" or "processing takes 10 minutes" means asynchronous inference, not real-time (25 MB, 60 s).
- Batch transform has no endpoint. "Lowest cost for scoring once a day" means batch transform.
- Multi-model endpoint needs the same container for all models. Different frameworks means multi-container.
- Fine-tuned Bedrock models usually need Provisioned Throughput. Cross-region profiles change routing, not price.

## Quiz

**1.** Claims documents of up to 300 MB must be scored within a few minutes of upload. Traffic stops at night
and the company wants to pay nothing then. Which option fits best?
A. Real-time endpoint  B. Serverless endpoint  C. Asynchronous endpoint with auto scaling to zero  D. Batch transform every hour
<details><summary>Answer</summary>C. Async takes payloads up to 1 GB, queues requests and can scale to 0 instances. Real-time and serverless payload limits are too small.</details>

**2.** A company has 400 XGBoost models, one per insurance broker. Most are called a few times a day.
What is the most cost-effective hosting?
A. 400 serverless endpoints  B. One multi-model endpoint  C. One multi-container endpoint  D. 400 production variants
<details><summary>Answer</summary>B. Same container, many artifacts, uneven traffic: multi-model endpoint loads models on demand. Multi-container is limited to 15 containers.</details>

**3.** Requests must first be transformed by a scikit-learn preprocessor and then scored by XGBoost, with one API call
and the same preprocessing code used for batch transform. What should you build?
A. Lambda in front of the endpoint  B. Serial inference pipeline  C. Multi-model endpoint  D. Two endpoints and Step Functions
<details><summary>Answer</summary>B. An inference pipeline chains 2 to 15 containers on one endpoint and also works with batch transform.</details>

**4.** An application calls a Bedrock model on demand and gets throttled at peak time. The team wants more
throughput without a commitment. What should they do?
A. Buy Provisioned Throughput for 6 months  B. Use a cross-region inference profile  C. Import the model  D. Move to a serverless SageMaker endpoint
<details><summary>Answer</summary>B. A cross-region inference profile spreads requests over several Regions at on-demand prices.</details>

**5.** Two open-weight LLMs each need one GPU. Traffic is low. The team wants both on one `ml.g5.12xlarge`
(4 GPUs) and wants to scale each model separately. Which feature?
A. Multi-model endpoint  B. Production variants  C. Inference components  D. Asynchronous inference
<details><summary>Answer</summary>C. Inference components give each model its own accelerators, memory and copy count on shared instances.</details>

## Save state

The serverless endpoint `safedrive-fraud` is used by later notebooks. Save its name and the model behind it.

In [ ]:
config_name = sm.describe_endpoint(EndpointName=ENDPOINT)["EndpointConfigName"]
variant = sm.describe_endpoint_config(EndpointConfigName=config_name)["ProductionVariants"][0]
saved = U.save_state(fraud_endpoint=ENDPOINT, fraud_model_name=variant["ModelName"])
{k: saved[k] for k in ["fraud_endpoint", "fraud_model_name"]}

## Cleanup

**Kept for later notebooks:** serverless endpoint `safedrive-fraud` with its model and endpoint config (no cost while idle).
Everything else is deleted here. Each section already deleted its own endpoint; these cells are a safety net.

In [ ]:
for name in ["safedrive-fraud-rt", "safedrive-fraud-async", "safedrive-fraud-mme",
             "safedrive-fraud-serial", "safedrive-fraud-sklearn", "safedrive-fraud-ic"]:
    if endpoint_status(name):
        U.safe(sm.delete_endpoint, EndpointName=name)
    if exists(sm.describe_endpoint_config, EndpointConfigName=name):
        U.safe(sm.delete_endpoint_config, EndpointConfigName=name)
print("instance endpoints left:", [e["EndpointName"] for e in sm.list_endpoints(NameContains="safedrive-fraud-")["Endpoints"]])

Delete the shared SageMaker model (models are free, this is only tidying up).

In [ ]:
U.safe(sm.delete_model, ModelName=MODEL)

Delete the application inference profile.

In [ ]:
U.safe(bedrock.delete_inference_profile, inferenceProfileIdentifier=app_arn)

Delete the S3 objects this notebook wrote (async, batch, multi-model and packaged artifacts).

In [ ]:
for prefix in ["safedrive/async/", "safedrive/batch/", "safedrive/mme/",
               "safedrive/pipeline-model/", "safedrive/external-model/"]:
    for key in U.list_keys(prefix):
        s3.delete_object(Bucket=BUCKET, Key=key)
    print("emptied", prefix)